### Dataset escogido
https://www.kaggle.com/datasets/laveshjadon/ai-impact-on-students/data

Dataset escogido para observar el comportamiento del rendimiento académico de muchos estudiantes que hacen uso de la IA generativa y como se refleja en su desempeño académico, desarrollo de habilidades y su salud mental

In [31]:
#Carga del dataset

import pandas as pd 
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv('ai_student_impact_dataset.csv') #carga del csv

print ("Dataset cargado")


Dataset cargado


In [32]:
df.head()

,Student_ID,Major_Category,Year_of_Study,Pre_Semester_GPA,Weekly_GenAI_Hours,Primary_Use_Case,Prompt_Engineering_Skill,Tool_Diversity,Paid_Subscription,Traditional_Study_Hours,Perceived_AI_Dependency,Institutional_Policy,Anxiety_Level_During_Exams,Post_Semester_GPA,Skill_Retention_Score,Burnout_Risk_Level
0,100001,Humanities,Senior,2.418,23.31,Copywriting/Drafting,Beginner,1,True,8.13,5,Allowed_With_Citation,6,2.393,86.44,High
1,100002,Medical,Junior,3.821,1.12,Ideation,Advanced,5,False,16.65,3,Allowed_With_Citation,9,3.696,69.39,Low
2,100003,Business,Freshman,3.398,21.26,Summarizing_Reading,Beginner,2,False,10.35,5,Strict_Ban,9,3.499,73.93,Medium
3,100004,Business,Senior,3.789,1.82,Copywriting/Drafting,Intermediate,4,False,15.23,2,Allowed_With_Citation,2,4.000,63.58,Medium
4,100005,STEM,Sophomore,3.635,9.29,Debugging/Troubleshooting,Advanced,4,False,12.55,4,Allowed_With_Citation,4,3.798,100.00,Medium


In [33]:
#consultar cuantas filas y colmnas existen con el .shape en el dataset
print("El dataset tiene", df.shape[0], "filas y", df.shape[1], "columnas")

El dataset tiene 50000 filas y 16 columnas


In [34]:
#Resumen de cada coluumna y sus valores nulos
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  50000 non-null  int64  
 1   Major_Category              50000 non-null  object 
 2   Year_of_Study               50000 non-null  object 
 3   Pre_Semester_GPA            50000 non-null  float64
 4   Weekly_GenAI_Hours          50000 non-null  float64
 5   Primary_Use_Case            50000 non-null  object 
 6   Prompt_Engineering_Skill    50000 non-null  object 
 7   Tool_Diversity              50000 non-null  int64  
 8   Paid_Subscription           50000 non-null  bool   
 9   Traditional_Study_Hours     50000 non-null  float64
 10  Perceived_AI_Dependency     50000 non-null  int64  
 11  Institutional_Policy        50000 non-null  object 
 12  Anxiety_Level_During_Exams  50000 non-null  int64  
 13  Post_Semester_GPA           500

In [35]:
#Buscamos datos faltantes en cada columna con el .isnull().sum()
df.isnull().sum()

Student_ID                    0
Major_Category                0
Year_of_Study                 0
Pre_Semester_GPA              0
Weekly_GenAI_Hours            0
Primary_Use_Case              0
Prompt_Engineering_Skill      0
Tool_Diversity                0
Paid_Subscription             0
Traditional_Study_Hours       0
Perceived_AI_Dependency       0
Institutional_Policy          0
Anxiety_Level_During_Exams    0
Post_Semester_GPA             0
Skill_Retention_Score         0
Burnout_Risk_Level            0
dtype: int64

In [36]:
#revisamos duplicados
duplicados = df.duplicated().sum()
print("El dataset tiene", duplicados, "filas duplicadas")

El dataset tiene 0 filas duplicadas


In [37]:
#Omitimos las columnas que no son necesarias para el analisis, las que no necesitan sacar promedios, y las que no tienen sentido para el analisis

columnas_categoricas = ['Major_Category', 'Year_of_Study', 'Primary_Use_Case',
                         'Prompt_Engineering_Skill', 'Institutional_Policy',
                         'Burnout_Risk_Level']

for columna in columnas_categoricas:
    print(f"--- {columna} ---")
    print(df[columna].unique())
    print()
    

--- Major_Category ---
['Humanities' 'Medical' 'Business' 'STEM' 'Arts']

--- Year_of_Study ---
['Senior' 'Junior' 'Freshman' 'Sophomore' 'Graduate']

--- Primary_Use_Case ---
['Copywriting/Drafting' 'Ideation' 'Summarizing_Reading'
 'Debugging/Troubleshooting' 'Direct_Answer_Generation']

--- Prompt_Engineering_Skill ---
['Beginner' 'Advanced' 'Intermediate']

--- Institutional_Policy ---
['Allowed_With_Citation' 'Strict_Ban' 'Actively_Encouraged']

--- Burnout_Risk_Level ---
['High' 'Low' 'Medium']



In [38]:
#Creamos nueva columna llamada 'GPA_Change' que contenga la diferencia entre 'GPA_After_AI' y 'GPA_Before_AI'

# Si el resultado es positivo, el estudiante mejoró su GPA.
# Si es negativo, el estudiante bajó su GPA.
df['GPA_Change'] = df['Post_Semester_GPA'] - df['Pre_Semester_GPA']

# Verificamos que la columna se haya creado correctamente.
df[['Pre_Semester_GPA', 'Post_Semester_GPA', 'GPA_Change']].head()

,Pre_Semester_GPA,Post_Semester_GPA,GPA_Change
0,2.418,2.393,-0.025
1,3.821,3.696,-0.125
2,3.398,3.499,0.101
3,3.789,4.000,0.211
4,3.635,3.798,0.163


###  Pregunta Número 1

#### ¿El uso intensivo de herramientas de IA generativa está asociado con cambios en el rendimiento académico (GPA)?

Para responder esto vamos a:
1. Clasificar a los estudiantes según cuántas horas semanales usan IA (bajo, medio, alto).
2. Comparar el cambio de GPA (`GPA_Change`) promedio entre esos grupos.
3. Revisar también si el nivel de habilidad usando prompts (`Prompt_Engineering_Skill`) influye.
4. Calcular la correlación entre las horas de uso y el cambio de GPA.

In [39]:
# Aquí dividimos las horas semanales de uso de IA en 3 grupos: Bajo, Medio, Alto.
# bins define los límites de cada rango, y labels el nombre que le queremos poner.

df['Nivel_Uso_IA'] = pd.cut(
    df['Weekly_GenAI_Hours'],
    bins=[-0.01, 5, 15, df['Weekly_GenAI_Hours'].max()],
    labels=['Bajo', 'Medio', 'Alto']
)

# Verificamos cuántos estudiantes cayeron en cada grupo.
df['Nivel_Uso_IA'].value_counts()

Nivel_Uso_IA
Bajo     22592
Medio    18874
Alto      8534
Name: count, dtype: int64

In [40]:
#Agrupamos por 'Nivel_Uso_IA' y calculamos el promedio de 'GPA_Change' para cada grupo.
cambio_gpa_por_uso = df.groupby('Nivel_Uso_IA', observed=True)['GPA_Change'].mean()
cambio_gpa_por_uso

Nivel_Uso_IA
Bajo     0.194685
Medio    0.227026
Alto     0.173030
Name: GPA_Change, dtype: float64

In [41]:
#Hacemos lo mismo pero agrupando por nivel de habilidad 
#usando los prompt engineering skill levels y calculando el promedio de 'Prompt_Engineering_Skill'.
#El objetivo es ver si se le da el uso adecuado a la IA, no si se usa mucho, sino si se usa bien.

cambio_gpa_por_skill = df.groupby('Prompt_Engineering_Skill', observed=True)['GPA_Change'].mean()
cambio_gpa_por_skill

Prompt_Engineering_Skill
Advanced        0.248096
Beginner        0.185244
Intermediate    0.186924
Name: GPA_Change, dtype: float64

In [42]:
# Combinamos las dos variables: Nivel_Uso_IA Y Prompt_Engineering_Skill juntas.
# .groupby() puede recibir una lista de columnas para cruzar varias categorías a la vez.
#
# Esto nos da una tabla más detallada: por ejemplo,
# "uso Alto + habilidad Avanzada" vs "uso Alto + habilidad Principiante".

tabla_cruzada = df.groupby(['Nivel_Uso_IA', 'Prompt_Engineering_Skill'], observed=True)['GPA_Change'].mean()
tabla_cruzada

Nivel_Uso_IA  Prompt_Engineering_Skill
Bajo          Advanced                    0.237730
              Beginner                    0.177975
              Intermediate                0.179277
Medio         Advanced                    0.271215
              Beginner                    0.208175
              Intermediate                0.212666
Alto          Advanced                    0.225741
              Beginner                    0.152134
              Intermediate                0.150318
Name: GPA_Change, dtype: float64

In [43]:
# .corr() calcula la correlación entre dos columnas numéricas.
# La correlación va de -1 a 1:
#   - Cerca de 1  -> cuando una sube, la otra también sube.
#   - Cerca de -1 -> cuando una sube, la otra baja.
#   - Cerca de 0  -> no hay relación clara entre las dos.

correlacion = df['Weekly_GenAI_Hours'].corr(df['GPA_Change'])
print("Correlación entre horas de uso de IA y cambio de GPA:", round(correlacion, 4))

Correlación entre horas de uso de IA y cambio de GPA: -0.0465


### Conclusión Pregunta 1
- La correlación entre horas semanales de uso de IA y el cambio de GPA es de **-0.0465**, es decir, prácticamente nula. Esto indica que **la cantidad de horas usando IA, por sí sola, no está asociada de forma relevante con que el GPA suba o baje**.
- En todos los grupos (Bajo, Medio, Alto uso) el cambio de GPA promedio fue positivo, pero el grupo de uso **Alto (0.173)** tuvo una mejora menor que el de uso **Medio (0.227)**, lo que sugiere que usar la IA en exceso no necesariamente potencia el rendimiento, e incluso podría ir asociado a un beneficio ligeramente menor.
- La variable que sí marca una diferencia más clara es el nivel de habilidad (`Prompt_Engineering_Skill`): los estudiantes **Avanzados** tuvieron el mayor cambio promedio de GPA (**0.248**), frente a Principiantes (**0.185**) e Intermedios (**0.187**).
- La tabla cruzada refuerza esto: en cada nivel de uso (Bajo, Medio o Alto), los estudiantes con habilidad Avanzada superan consistentemente a los Beginner/Intermediate. Por ejemplo, en uso Alto, Avanzado llega a 0.226 mientras Beginner cae a 0.152.
- **Conclusión general:** no es *cuánto* se usa la IA lo que más se relaciona con la mejora del GPA, sino *qué tan bien se sabe usar* (habilidad de prompting). Esto es coherente con la idea de que la IA es una herramienta que potencia el aprendizaje solo cuando se usa con criterio, y no un sustituto automático del estudio.

#### Pregunta Número 2: 2.	¿Existe relación entre la dependencia percibida de la IA (Perceived_AI_Dependency) y el riesgo de burnout o los niveles de ansiedad en exámenes?

Vamos a:
1. Ver si a mayor dependencia percibida de la IA (\`Perceived_AI_Dependency\`), sube el nivel de ansiedad en exámenes.
2. Cruzar la dependencia percibida con el riesgo de burnout (\`Burnout_Risk_Level\`).
3. Calcular la correlación entre dependencia percibida y ansiedad.

In [44]:
# Agrupamos por 'Perceived_AI_Dependency' (un puntaje del 1 al 10, por ejemplo)
# y calculamos el promedio de ansiedad en exámenes para cada nivel de dependencia.
#
# Esto nos dice: a medida que un estudiante siente que depende más de la IA,
# ¿su ansiedad promedio en exámenes sube o baja?

ansiedad_por_dependencia = df.groupby('Perceived_AI_Dependency', observed=True)['Anxiety_Level_During_Exams'].mean()
ansiedad_por_dependencia

Perceived_AI_Dependency
1     3.423388
2     3.741492
3     4.068334
4     4.390779
5     4.782061
6     5.187447
7     5.610337
8     6.042105
9     6.423280
10    6.915789
Name: Anxiety_Level_During_Exams, dtype: float64

In [45]:
# pd.crosstab() cruza dos columnas categóricas y cuenta cuántos casos hay
# en cada combinación. Aquí cruzamos el nivel de dependencia percibida
# con el riesgo de burnout, para ver si van de la mano.
#
# normalize='index' convierte los conteos en porcentajes por fila,
# así es más fácil comparar (en vez de números absolutos).

cruce_dependencia_burnout = pd.crosstab(
    df['Perceived_AI_Dependency'],
    df['Burnout_Risk_Level'],
    normalize='index'
).round(3) * 100  # lo pasamos a porcentaje

cruce_dependencia_burnout

Burnout_Risk_Level,High,Low,Medium
Perceived_AI_Dependency,,,
1,9.9,49.0,41.0
2,13.4,42.6,44.0
3,18.3,36.9,44.8
4,24.1,30.4,45.5
5,32.9,22.9,44.2
6,48.5,13.4,38.2
7,66.4,4.9,28.7
8,79.5,2.3,18.1
9,86.2,1.3,12.4


In [46]:
# Calculamos la correlación entre dependencia percibida de la IA y ansiedad en exámenes.
# Recordemos: cerca de 1 o -1 = relación fuerte, cerca de 0 = relación débil o nula.

correlacion_dependencia_ansiedad = df['Perceived_AI_Dependency'].corr(df['Anxiety_Level_During_Exams'])
print("Correlación entre dependencia percibida de IA y ansiedad en exámenes:",
      round(correlacion_dependencia_ansiedad, 4))

Correlación entre dependencia percibida de IA y ansiedad en exámenes: 0.3076


### Conclusión pregunta 2

- La ansiedad promedio en exámenes sube de forma **clara y constante** a medida que aumenta la dependencia percibida de la IA: pasa de **3.42** (dependencia = 1) a **6.92** (dependencia = 10), prácticamente el doble.
- La correlación entre ambas variables es **0.3076**, positiva y moderada — confirma que hay una relación real, aunque no es la única variable que explica la ansiedad (no es una correlación fuerte tipo 0.7+).
- El cruce con \`Burnout_Risk_Level\` es aún más contundente: con dependencia baja (1), solo el **9.9%** de los estudiantes tiene riesgo de burnout Alto. Con dependencia alta (10), ese porcentaje sube a **94.7%**, mientras que el riesgo Bajo prácticamente desaparece (de 49% a 0%).
- **Conclusión general:** entre más dependiente se percibe un estudiante de la IA, más ansiedad muestra en exámenes y mucho mayor es su riesgo de burnout. Esto sugiere que la dependencia (no el uso en sí) está más ligada al desgaste emocional que a un beneficio académico — apoyando la idea de que usar la IA como muleta constante, en vez de como apoyo puntual, tiene un costo en bienestar del estudiante.

### Pregunta 3: ¿Cómo varía el uso de IA y su impacto según la carrera y la política institucional?

Vamos a:
1. Comparar el uso de IA, la retención de habilidades y el GPA final entre carreras (\`Major_Category\`).
2. Comparar esas mismas métricas según la política institucional (\`Institutional_Policy\`).
3. Cruzar carrera y política juntas para ver el detalle.

In [47]:
# Agrupamos por carrera (Major_Category) y calculamos el promedio de varias
# columnas a la vez usando .agg(), que nos permite resumir varias métricas
# en una sola tabla.

resumen_por_carrera = df.groupby('Major_Category', observed=True).agg(
    Horas_Uso_IA_Prom=('Weekly_GenAI_Hours', 'mean'),
    Retencion_Habilidad_Prom=('Skill_Retention_Score', 'mean'),
    GPA_Final_Prom=('Post_Semester_GPA', 'mean')
).round(2)

resumen_por_carrera

,Horas_Uso_IA_Prom,Retencion_Habilidad_Prom,GPA_Final_Prom
Major_Category,,,
Arts,7.27,75.66,3.34
Business,8.28,75.26,3.34
Humanities,6.77,75.25,3.35
Medical,7.55,75.48,3.35
STEM,10.49,76.80,3.36


In [48]:
# Hacemos lo mismo, pero agrupando por la política institucional
# (si la universidad prohíbe, permite con citación, o fomenta el uso de IA).

resumen_por_politica = df.groupby('Institutional_Policy', observed=True).agg(
    Horas_Uso_IA_Prom=('Weekly_GenAI_Hours', 'mean'),
    Retencion_Habilidad_Prom=('Skill_Retention_Score', 'mean'),
    GPA_Final_Prom=('Post_Semester_GPA', 'mean')
).round(2)

resumen_por_politica

,Horas_Uso_IA_Prom,Retencion_Habilidad_Prom,GPA_Final_Prom
Institutional_Policy,,,
Actively_Encouraged,8.48,76.13,3.35
Allowed_With_Citation,8.42,75.91,3.35
Strict_Ban,8.37,74.99,3.33


In [49]:
# Ahora cruzamos carrera y política institucional al mismo tiempo,
# para ver si el efecto de la política cambia según la carrera.
# pivot_table es ideal para esto: arma automáticamente una tabla
# tipo Excel con carreras en filas y políticas en columnas.

pivote_retencion = df.pivot_table(
    values='Skill_Retention_Score',
    index='Major_Category',
    columns='Institutional_Policy',
    aggfunc='mean',
    observed=True
).round(2)

pivote_retencion

Institutional_Policy,Actively_Encouraged,Allowed_With_Citation,Strict_Ban
Major_Category,,,
Arts,75.73,75.91,74.93
Business,75.61,75.34,74.48
Humanities,75.47,75.38,74.56
Medical,75.65,75.72,74.61
STEM,77.42,76.82,75.85


In [50]:
# Filtramos: nos quedamos solo con los estudiantes de carreras STEM
# que están bajo una política de "Strict_Ban" (prohibición estricta),
# para explorar un caso puntual con un filtro simple.
#
# Esto es un ejemplo de cómo aplicar filtros combinados con &.

filtro_stem_prohibido = df[(df['Major_Category'] == 'STEM') & (df['Institutional_Policy'] == 'Strict_Ban')]

print("Cantidad de estudiantes STEM con política de prohibición estricta:", len(filtro_stem_prohibido))
print("Retención de habilidad promedio en este grupo:", round(filtro_stem_prohibido['Skill_Retention_Score'].mean(), 2))
print("GPA final promedio en este grupo:", round(filtro_stem_prohibido['Post_Semester_GPA'].mean(), 2))

Cantidad de estudiantes STEM con política de prohibición estricta: 3043
Retención de habilidad promedio en este grupo: 75.85
GPA final promedio en este grupo: 3.35


### Conclusión Pregunta 3

- Por **carrera**, las diferencias son pequeñas pero consistentes: STEM es la carrera que más usa IA (**10.49 hrs/semana**, claramente por encima del resto) y también la que tiene mejor retención de habilidad (**76.80**) y mejor GPA final (**3.36**). Humanidades es la que menos usa la IA (**6.77 hrs**).
- Por **política institucional**, el patrón es claro y va en la misma dirección en las tres métricas: \`Actively_Encouraged\` (76.13 retención) > \`Allowed_With_Citation\` (75.91) > \`Strict_Ban\` (74.99, la más baja). Es decir, **prohibir estrictamente el uso de IA no mejora la retención de habilidades ni el GPA**; de hecho, el grupo con prohibición estricta queda último en ambas métricas.
- La tabla cruzada (pivote) muestra que este patrón se mantiene **dentro de cada carrera**: en STEM, por ejemplo, la retención baja de 77.42 (Fomentado) a 75.85 (Prohibido). Lo mismo ocurre en las demás carreras, aunque con diferencias más chicas.
- **Conclusión general:** ni la carrera ni la política institucional generan diferencias dramáticas por sí solas, pero la tendencia es consistente: los entornos que **fomentan o permiten el uso de IA con citación** logran retención de habilidad y GPA ligeramente mejores que los que la prohíben estrictamente. Esto sugiere que **prohibir la IA no protege el aprendizaje** — probablemente porque los estudiantes la usan de todas formas, solo que sin guía ni transparencia.

## 6. Conclusiones generales del taller

Uniendo las tres preguntas, el panorama general del dataset es el siguiente:

1. **No es cuánto se usa la IA lo que importa, sino cómo se usa.** El tiempo de uso por sí solo casi no se correlaciona con el cambio de GPA (correlación de -0.05), pero la habilidad de prompting sí marca diferencia (Avanzados mejoran 0.25 puntos más que Principiantes).

2. **La dependencia (no el uso) es lo que más se relaciona con el desgaste emocional.** A mayor dependencia percibida, la ansiedad en exámenes casi se duplica y el riesgo de burnout Alto pasa de 9.9% a 94.7% — la relación más fuerte encontrada en todo el análisis.

3. **Prohibir la IA no parece ser la mejor estrategia institucional.** Los grupos bajo políticas que fomentan o permiten el uso (con citación) muestran, de forma consistente, mejor retención de habilidades y GPA que los grupos con prohibición estricta — tanto a nivel general como dentro de cada carrera.

**En conjunto**, estos hallazgos apuntan a que el reto no es decidir si los estudiantes deben o no usar IA, sino **enseñarles a usarla bien** (habilidad de prompting) y **evitar que se vuelvan dependientes de ella**, en vez de simplemente prohibirla.